In [ ]:
#Dependinces for helping us to beat the facebook detector for
!pip install playwright nest_asyncio
!playwright install chromium
!playwright install-deps
!pip install azure-cosmos python-dotenv
!pip install langdetect


In [ ]:
import json
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from langdetect import detect, DetectorFactory
import os
import uuid
from dotenv import load_dotenv
from azure.cosmos import CosmosClient
import random



DetectorFactory.seed = 0
nest_asyncio.apply()



async def scrape_fb_comments(video_url, max_comments=300):
    results_list = []


    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 720}
        )

        try:
            with open('fb_cookies.json', 'r') as f:
                cookies = json.load(f)
                await context.add_cookies(cookies)
                print("Cookies loaded!")
        except:
            print("Cookie file missing!")
            await browser.close()
            return []

        page = await context.new_page()


        try:
            print(f"Go to: {video_url}")
            await page.goto(video_url, wait_until="domcontentloaded", timeout=60000)
            await asyncio.sleep(1.5)

            #Open comments panel
            try:
                comment_btn = await page.wait_for_selector('div[aria-label="Comentează"], div[aria-label="Comments"]', timeout=7000)
                if comment_btn:
                    await comment_btn.click()
                    print("Sidebar opened.")
                    await asyncio.sleep(1)
            except:
                print("Sidebar might be already open or button changed.")

            comments_data = set()
            attempts_with0 = 0
            attempts = 0
            print("Searching for comments in sidebar...")
            while len(comments_data) < max_comments and attempts < 30:

                #SCROLL
                await page.evaluate('''
                    const comments = document.querySelectorAll('div[role="article"]');
                    if (comments.length > 0) {
                        comments[comments.length - 1].scrollIntoView({behavior: "smooth", block: "end"});
                    }
                ''')
                await asyncio.sleep(0.7)

                #See more - entire comment
                await page.evaluate('''
                    const buttons = document.querySelectorAll('div[role="button"]');
                    buttons.forEach(btn => {
                        if(btn.innerText && (btn.innerText.includes('Vezi mai mult') || btn.innerText.includes('See more'))) {
                            btn.click();
                        }
                    });
                ''')
                await asyncio.sleep(0.3)

                #Comments extraction
                comment_blocks = await page.query_selector_all('div[role="article"]')

                new_found = 0
                for block in comment_blocks:
                    try:
                        text_element = await block.query_selector('div[dir="auto"]')

                        if text_element:
                            #For emojis etc
                            text = await text_element.evaluate('''el => {
                                let ext = "";
                                function traverse(node) {
                                    if (node.nodeType === Node.TEXT_NODE) {
                                        ext += node.textContent;
                                    } else if (node.nodeName === "IMG" && node.hasAttribute("alt")) {
                                        ext += node.getAttribute("alt");
                                    } else {
                                        for (let child of node.childNodes) {
                                            traverse(child);
                                        }
                                    }
                                }
                                traverse(el);
                                return ext;
                            }''')
                            clean_text = text.strip()

                            if len(clean_text) > 0 and clean_text not in comments_data:
                                if detect(clean_text) == 'en':
                                    comments_data.add(clean_text)
                                    new_found += 1
                    except:
                        continue

                if new_found == 0:
                    attempts_with0 += 1
                if attempts_with0 == 3:
                    break

                attempts += 1

            print("------------------------------------------------")
            results_list = list(comments_data)[:max_comments]

        except Exception as e:
            print(f"An error occurred to the route to page: {e}")
        finally:
            await browser.close()


    return results_list



async def scrape_fb_reels(category, category_urls, target_per_categorie=3000):
    if 'AZURE_COSMOS_CONNECTION_STRING' in os.environ:
      del os.environ['AZURE_COSMOS_CONNECTION_STRING']
    load_dotenv('/content/.env')

    CONNECTION_STRING = os.getenv('AZURE_COSMOS_CONNECTION_STRING')
    if not CONNECTION_STRING:
        print("No  AZURE_COSMOS_CONNECTION_STRING in .env!")
        return


    try:
        client = CosmosClient.from_connection_string(CONNECTION_STRING)
        database = client.get_database_client('base_nlp')
        c3_fb_container = database.get_container_client('C2_FB')
        video_container = database.get_container_client('Video')
    except Exception as e:
        print(f"Error at conection to Azure: {e}")
        return


    #Go through categories
    used_urls = set()
    query = "SELECT c.video_url FROM c"

    items = list(video_container.query_items(
        query=query,
        enable_cross_partition_query=True
    ))
    for item in items:
        if 'video_url' in item:
            used_urls.add(item['video_url'])


    nr =  0
    print(f"\n{'='*40}")
    print(f"PROCESSING : {category.upper()}")
    print(f"{'='*40}")

    required_comments = target_per_categorie

    #Processing each reel
    for url in category_urls:
            if url in used_urls:
                continue
            else:
                used_urls.add(url)

            if required_comments == 0:
                break

            video_id = str(uuid.uuid4())
            try:
              with open("video.txt", "a", encoding="utf-8") as g:
                  g.write(f"{video_id} {url} {category}"'\n')
            except:
              print("Error at writing in video.txt")

            #Extract the comments
            extracted_comments = await scrape_fb_comments(url, max_comments= min(required_comments, 300))
            nr = nr + len(extracted_comments)
            required_comments = required_comments - len(extracted_comments)

            try:
              with open("comments_fb.jsonl", "a", encoding="utf-8") as h:
                  for text in extracted_comments:
                      comment_data = {"video_id": video_id, "text": text}
                      h.write(json.dumps(comment_data) + '\n')
            except:
              print("Error at writing in comments_fb.jsonl")

            pause = random.uniform(2, 4)
            await asyncio.sleep(pause)#For human behaviour

    print(f"Category {category}: "+str(nr))
    print("------------------------------------------------")
    print("------------------------------------------------")
    print("------------------------------------------------")



d = {"Game.txt": "gaming", "Tech.txt": "tech", "Education.txt": "education", "Sport.txt":"sport", "Comedy.txt":"comedy"}
cat = "Comedy.txt"
try:
      with open(cat, 'r', encoding="utf-8") as f:
                urls = [line.strip() for line in f.readlines() if line.strip()]
except FileNotFoundError:
      print(f"Problem with file {cat}")


asyncio.run(scrape_fb_reels(d[cat], urls))



In [8]:
import json
import os
import uuid
from dotenv import load_dotenv
from azure.cosmos import CosmosClient


if 'AZURE_COSMOS_CONNECTION_STRING' in os.environ:
      del os.environ['AZURE_COSMOS_CONNECTION_STRING']
load_dotenv('/content/.env')

CONNECTION_STRING = os.getenv('AZURE_COSMOS_CONNECTION_STRING')
if not CONNECTION_STRING:
      print("No  AZURE_COSMOS_CONNECTION_STRING in .env!")


try:
      client = CosmosClient.from_connection_string(CONNECTION_STRING)
      database = client.get_database_client('base_nlp')
      c3_fb_container = database.get_container_client('C2_FB')
      video_container = database.get_container_client('Video')
except Exception as e:
      print(f"Error at conection to Azure: {e}")

try:
  with open("video.txt", "r", encoding="utf-8") as f:
        for line in f:
          elem = line.strip().split()
          video_container.upsert_item({
                "id": elem[0],
                "video_url": elem[1],
                "category": elem[2],
          })
except:
  print("Error at reading from video.txt")

try:
  with open("comments_fb.jsonl", "r", encoding="utf-8") as f:
                      for line in f:
                          data = json.loads(line.strip())
                          comment = {
                                "id": str(uuid.uuid4()),
                                "text": data["text"],
                                "label": 1,
                                "video_id": data["video_id"]
                          }
                          c3_fb_container.upsert_item(comment)
except:
  print("Error at reading from comments_fb.jsonl")


In [ ]:
import json
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from langdetect import detect, DetectorFactory
import os
import uuid
from dotenv import load_dotenv
from azure.cosmos import CosmosClient
import random



if 'AZURE_COSMOS_CONNECTION_STRING' in os.environ:
    del os.environ['AZURE_COSMOS_CONNECTION_STRING']
load_dotenv('/content/.env')

CONNECTION_STRING = os.getenv('AZURE_COSMOS_CONNECTION_STRING')
if not CONNECTION_STRING:
        print("No  AZURE_COSMOS_CONNECTION_STRING in .env!")

try:
        client = CosmosClient.from_connection_string(CONNECTION_STRING)
        database = client.get_database_client('base_nlp')
        c3_fb_container = database.get_container_client('C2_FB')
        video_container = database.get_container_client('Video')
except Exception as e:
        print(f"Error at conection to Azure: {e}")




used_urls = set()
query = "SELECT c.video_url FROM c"

items = list(video_container.query_items(
        query=query,
        enable_cross_partition_query=True
))
for item in items:
        if 'video_url' in item:
            used_urls.add(item['video_url'])
print(used_urls)

{'https://www.youtube.com/shorts/207795c7-8229-419b-b1cc-e67a608550bf', 'https://www.tiktok.com/@balouskiin/video/7625683176744078624', 'https://www.youtube.com/shorts/92be6a5a-a9af-48b6-9ee8-1420d6637654', 'https://www.youtube.com/shorts/6ddeb405-763c-4e69-933b-5d292fe5c549', 'https://www.youtube.com/shorts/9a4818a3-8839-4b52-a36c-c6ffc6484ac2', 'https://www.tiktok.com/@niickjackson/video/7619530585454808334', 'https://www.tiktok.com/@houseofhighlights/video/7589118587953024270', 'https://www.youtube.com/shorts/1b89e60f-364c-4d81-951c-85bb9f37140f', 'https://www.tiktok.com/@dailylols7/video/7623070341316775199', 'https://www.youtube.com/shorts/cc463db6-bfa8-486a-89a3-b13878d94f79', 'https://www.tiktok.com/@astroathens/video/6808960524206869766', 'https://www.youtube.com/shorts/daff28a7-d486-4a05-9fee-fbd28ed6aa98', 'https://www.youtube.com/shorts/f28fb3c5-5ded-4b95-9a8b-1a79bf76e8e6', 'https://www.youtube.com/shorts/2e6c9caa-79ca-4968-8aa4-bd89258d7487', 'https://www.youtube.com/short